In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules/python-utils")

In [ ]:
import librosa
import numpy as np

In [ ]:
from rt_whisper.streamers import get_token_streamer

In [ ]:
SAMPLE_RATE = 16000

In [ ]:
audio, sr = librosa.load(
    "/workspaces/dev/.data/news_with_English.mp3",
    sr=SAMPLE_RATE
)

In [ ]:
# audio = audio[85 * SAMPLE_RATE:]

In [ ]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
    rand_len = int(np.random.normal(loc=48000, scale=400))
    rand_len = np.clip(rand_len, 46000, 50000)
    end = min(pos + rand_len, total_samples)
    # end = min(pos + 16000, total_samples)

    chunk = audio[pos:end]
    segments.append(chunk)
    pos = end

In [ ]:
token_streamer = get_token_streamer()

In [ ]:
raise Exception("stop")

In [ ]:
from rt_whisper.data import Param, Result
from IPython.display import Audio

In [ ]:
segment_idx = 0
completed = []
param = Param()
param.offset = 0

In [ ]:
segment = segments[segment_idx]
segment_idx += 1

param.chunk = segment

result:Result = token_streamer.process(param)
completed.extend(result.completed)

print(f"{segment_idx}" + "--" * 20)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.completed]
)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.candidate]
)

param.update(result)


In [ ]:
from rt_whisper.processors.asr import ASRState
Audio(result.context_dict[ASRState].chunk, rate=SAMPLE_RATE)

In [ ]:
from rt_whisper.processors.vad import VADState
# VAD
Audio(result.context_dict[VADState].chunk, rate=SAMPLE_RATE)

In [ ]:
start = result.completed[0].tokens[0].start
end = result.completed[0].tokens[-1].end
Audio(audio[start: end], rate=SAMPLE_RATE)

In [ ]:
start = result.completed[1].tokens[0].start
end = result.completed[1].tokens[-1].end
Audio(audio[start: end], rate=SAMPLE_RATE)

In [ ]:
Audio(result.recycles["vad"].vad_chunk, rate=SAMPLE_RATE)
# len(result.recycles["vad"].vad_chunk), len(segment)

In [ ]:
for i, segment in enumerate(segments):
    param.chunk = segment

    result:Result = token_streamer.process(param)
    completed.extend(result.completed)

    print(f"{i}" + "--" * 20)
    # print([(v.lang, v.text) for v in completed])
    print([(v.lang, v.text) for v in result.completed])
    print([(v.lang, v.text) for v in result.candidate])
    # print([(v.lang, v.text) for v in result.prev_completed_tokens if v.is_word])
    # print([(v.lang, v.text) for v in result.prev_candidate_tokens if v.is_word])

    param.update(result)

In [ ]:
for v in completed:
    print(v.lang, v.text)